# Fine-tuning YOLOP for BeamNG control correction, using Comma2k19

This notebook fine-tunes the project's YOLOP model (the official `YOLOP/lib/models/yolop.py`
architecture, pretrained weights in `YOLOP/weights/End-to-end.pth`) so that, given a dashcam
frame plus the car's *current* acceleration/brake/steer/speed, it predicts *corrected*
acceleration/brake/steer to drive in BeamNG.

The YOLOP backbone is kept frozen (same spirit as the LoRA fine-tuning in the original version
of this notebook: keep the big pretrained model frozen, train only a small new part). A small
fusion head is trained on top, using [Comma2k19](https://github.com/commaai/comma2k19) — a
real-world driving dataset — as the source of (image, telemetry) -> (future telemetry) examples.

Comma2k19 has no explicit brake/gas pedal channel, so acceleration/brake are derived from the
IMU's longitudinal acceleration and converted into BeamNG's `throttle`/`brake` (0..1) via
`comma_beamng_transforms.py`. Steering wheel angle (degrees) is converted to BeamNG's
`steering` (-1..1) the same way. These transform functions are written once and are directly
reusable by `yolop_beamng.py` later.

Pipeline: setup -> download a small Comma2k19 subset -> build a (frame, telemetry) manifest ->
load the frozen YOLOP base model -> train the new fusion head -> evaluate -> sanity-check
inference.

In [8]:
%pip uninstall -y torch torchvision torchaudio -q

# Plain `pip install torch` on Windows resolves to the CPU-only wheel. Install the
# CUDA build explicitly so torch.cuda.is_available() picks up the local GPU. cu126
# works with any driver supporting CUDA >= 12.6 (check with `nvidia-smi`); bump the
# index URL (e.g. cu128, cu129) if pip can't find a wheel for your Python version.
%pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu126
%pip install -q opencv-python imageio-ffmpeg pandas numpy

# Optional: only needed for tools/download_comma2k19.py's automated subset download.
# If it fails to install (common on Windows), the download script falls back to
# printing manual instructions for a regular BitTorrent client.
%pip install -q libtorrent

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement libtorrent (from versions: none)
ERROR: No matching distribution found for libtorrent


In [9]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
device = "cuda" if torch.cuda.is_available() else "cpu"

CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Ti


## 1. Download a small Comma2k19 subset

The full dataset is ~100GB over BitTorrent. The torrent itself only contains 10 whole-chunk
zip files (`Chunk_1.zip` .. `Chunk_10.zip`, ~9-10GB each) -- BitTorrent can only select whole
files, so there's no way to fetch individual segments over the wire despite the per-segment
folder layout documented in the upstream README (that layout describes what's *inside* each
zip, not the torrent's own file list).

So `tools/download_comma2k19.py` downloads exactly one whole chunk zip, then extracts only
`--max-segments` segment folders from inside it locally (without unpacking the other ~200
segments in the same chunk), and deletes the zip afterward to reclaim the ~9GB.

It tries `libtorrent` first, then falls back to driving a running **qBittorrent**'s Web UI
(via `qbittorrent-api`) -- only this fallback worked in this environment, since `libtorrent`
has no wheel for Windows + Python 3.14. One-time setup if you haven't already:

1. Install qBittorrent (already done: `winget install qBittorrent.qBittorrent`) and make sure
   it's actually **running** (it doesn't auto-start).
2. Open it once, go to **Tools > Options > Web UI**, check "Web User Interface", note the
   port (default 8080), and set a username/password.
3. Set `QBT_USERNAME` / `QBT_PASSWORD` below to match.

If qBittorrent isn't reachable either, the script prints manual instructions and exits with
an error instead of silently leaving `RAW_DIR` empty.

In [10]:
import os
from pathlib import Path

RAW_DIR = "data/comma2k19/raw"

os.environ["QBT_USERNAME"] = "admin"      # match what you set in qBittorrent's Web UI options
os.environ["QBT_PASSWORD"] = "CHANGE_ME"  # do not commit a real password into the notebook

!python tools/download_comma2k19.py --out {RAW_DIR} --chunks 1 --max-segments 15

n_segments = len(list(Path(RAW_DIR).rglob("video.hevc")))
assert n_segments > 0, (
    f"No segments downloaded into {RAW_DIR} -- check the error/instructions printed above "
    "(likely qBittorrent Web UI credentials or connectivity) before continuing."
)
print(f"\n{n_segments} segment(s) ready in {RAW_DIR}")


No automated download method worked in this environment.
To download a small subset of comma2k19 manually:
  1. Install a BitTorrent client (qBittorrent or Transmission).
  2. Open this magnet link:
     magnet:?xt=urn:btih:65a2fbc964078aff62076ff4e103f18b951c5ddb&dn=comma2k19&tr=udp://tracker.openbittorrent.com:80/announce&tr=udp://tracker.opentrackr.org:1337/announce&tr=http://academictorrents.com/announce.php
  3. The torrent only contains whole chunk zips (Chunk_1.zip .. Chunk_10.zip, ~9-10GB each) -- select only Chunk_1.zip.
  4. Set the download location to: C:\Users\nalli\Documents\Workspace Uni\TP FINAL Vision\TP-final-Vision-Artificial\data\comma2k19\raw
  5. Once it finishes, extract just a few segment folders from the zip yourself, or re-run this script with --keep-zip pointed at the downloaded file via --out, and it will do the extraction step for you (skip straight to that by placing the zip at C:\Users\nalli\Documents\Workspace Uni\TP FINAL Vision\TP-final-Vision-Artific

libtorrent not installed, trying qBittorrent Web UI...
qBittorrent Web UI download failed: 


## 2. Inspect one segment's `processed_log` layout

comma2k19 mirrors have varied slightly in exact `processed_log` sub-paths (e.g.
`CAN/car_speed` vs `CAN/speed`). Before building the full manifest, inspect one downloaded
segment to confirm the signal lookup in `tools/prepare_comma2k19_control_dataset.py`
(fuzzy-matched by keyword tokens) is actually finding the speed/steering/IMU arrays. If it
isn't, adjust `SIGNAL_TOKENS` in that script to match what's printed below.

In [11]:
segments = list(Path(RAW_DIR).rglob("video.hevc"))
assert segments, f"No video.hevc found under {RAW_DIR} -- re-run the download cell above first."

first_segment = segments[0].parent
print(f"Inspecting: {first_segment}\n")

!python tools/prepare_comma2k19_control_dataset.py --inspect "{first_segment}"

Inspecting: data\comma2k19\raw\Chunk_1\b0c9d2329ad1606b_2018-08-03--10-35-16\11

Segment: data\comma2k19\raw\Chunk_1\b0c9d2329ad1606b_2018-08-03--10-35-16\11
  global_pose\frame_gps_times  shape=(1199, 2) dtype=float64
  global_pose\frame_orientations  shape=(1199, 4) dtype=float64
  global_pose\frame_positions  shape=(1199, 3) dtype=float64
  global_pose\frame_times  shape=(1199,) dtype=float64
  global_pose\frame_velocities  shape=(1199, 3) dtype=float64
  preview.png
  processed_log\CAN\radar\t  shape=(11612,) dtype=float64
  processed_log\CAN\radar\value  shape=(11612, 7) dtype=float64
  processed_log\CAN\raw_can\address  shape=(135472,) dtype=int64
  processed_log\CAN\raw_can\data  shape=(135472,) dtype=|S8
  processed_log\CAN\raw_can\src  shape=(135472,) dtype=int64
  processed_log\CAN\raw_can\t  shape=(135472,) dtype=float64
  processed_log\CAN\speed\t  shape=(4973,) dtype=float64
  processed_log\CAN\speed\value  shape=(4973, 1) dtype=float64
  processed_log\CAN\steering_angle\t

## 3. Build the training manifest

Extracts frames from each segment's `video.hevc`, aligns speed/steering/IMU-acceleration to
each frame and to a future horizon (`--horizon-seconds`, default 1.0s), applies the
comma2k19 -> BeamNG transforms, and writes `data/comma2k19/manifest.csv` with a per-segment
train/dev/test split (70/15/15) — split by segment, not by frame, so adjacent near-duplicate
frames don't leak across splits.

In [12]:
FRAMES_DIR = "data/comma2k19/frames"
MANIFEST_PATH = "data/comma2k19/manifest.csv"

!python tools/prepare_comma2k19_control_dataset.py --raw-dir {RAW_DIR} --frames-dir {FRAMES_DIR} --manifest {MANIFEST_PATH} --horizon-seconds 1.0

Processing Chunk_1_b0c9d2329ad1606b_2018-07-30--13-44-30_10...
  1181 usable frames
Processing Chunk_1_b0c9d2329ad1606b_2018-07-30--13-44-30_11...
  1181 usable frames
Processing Chunk_1_b0c9d2329ad1606b_2018-07-30--13-44-30_12...
  1181 usable frames
Processing Chunk_1_b0c9d2329ad1606b_2018-07-30--13-44-30_13...
  1181 usable frames
Processing Chunk_1_b0c9d2329ad1606b_2018-07-30--13-44-30_14...
  1181 usable frames
Processing Chunk_1_b0c9d2329ad1606b_2018-07-30--13-44-30_15...
  1181 usable frames
Processing Chunk_1_b0c9d2329ad1606b_2018-07-30--13-44-30_6...
  1181 usable frames
Processing Chunk_1_b0c9d2329ad1606b_2018-07-30--13-44-30_7...
  1181 usable frames
Processing Chunk_1_b0c9d2329ad1606b_2018-07-30--13-44-30_8...
  1181 usable frames
Processing Chunk_1_b0c9d2329ad1606b_2018-07-30--13-44-30_9...
  1181 usable frames
Processing Chunk_1_b0c9d2329ad1606b_2018-08-03--10-35-16_11...
  1181 usable frames
Processing Chunk_1_b0c9d2329ad1606b_2018-08-03--10-35-16_12...
  1182 usable fra

In [13]:
import pandas as pd

manifest = pd.read_csv(MANIFEST_PATH)
print(manifest["split"].value_counts())
print()
print(manifest[["throttle_t", "brake_t", "steer_t", "speed_t",
                "throttle_target", "brake_target", "steer_target"]].describe())

split
train    11811
test      3542
dev       2361
Name: count, dtype: int64

         throttle_t       brake_t       steer_t       speed_t  \
count  17714.000000  17714.000000  17714.000000  17714.000000   
mean       0.012067      0.103832     -0.001588     21.514151   
std        0.055901      0.073049      0.008781      9.926114   
min        0.000000      0.000000     -0.061000      0.000000   
25%        0.000000      0.054419     -0.004000     13.813715   
50%        0.000000      0.097208     -0.000200     26.742361   
75%        0.000000      0.145662      0.002600     29.231944   
max        1.000000      0.749552      0.027000     33.481250   

       throttle_target  brake_target  steer_target  
count     17714.000000  17714.000000  17714.000000  
mean          0.012578      0.103726     -0.001530  
std           0.058444      0.073208      0.008868  
min           0.000000      0.000000     -0.061000  
25%           0.000000      0.054138     -0.004000  
50%           0.00

## 4. Load the frozen YOLOP base model + new fusion head

`YOLOPControlNet` (in `control_finetune.py`) loads `YOLOP/lib/models/yolop.py` with
`YOLOP/weights/End-to-end.pth` (same pattern as `yolop_reproduction.ipynb`), freezes every
backbone parameter, and adds a small trainable fusion head: a forward hook on the shared
encoder layer (`model.model[16]`, 256 channels) feeds pooled image features into an MLP
together with the current `[throttle, brake, steer, speed]`, predicting corrected
`[throttle, brake, steer]`.

In [14]:
from control_finetune import YOLOPControlNet

control_model = YOLOPControlNet(device=device).to(device)

n_trainable = sum(p.numel() for p in control_model.trainable_parameters())
n_frozen = sum(p.numel() for p in control_model.backbone.parameters())
print(f"Trainable (fusion head + numeric branch): {n_trainable:,}")
print(f"Frozen (YOLOP backbone):                  {n_frozen:,}")

Trainable (fusion head + numeric branch): 46,659
Frozen (YOLOP backbone):                  7,940,846


## 5. Datasets and dataloaders

In [15]:
from torch.utils.data import DataLoader
from control_finetune import Comma2k19ControlDataset

train_ds = Comma2k19ControlDataset(MANIFEST_PATH, FRAMES_DIR, split="train")
dev_ds = Comma2k19ControlDataset(MANIFEST_PATH, FRAMES_DIR, split="dev")
test_ds = Comma2k19ControlDataset(MANIFEST_PATH, FRAMES_DIR, split="test")

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=0)
dev_loader = DataLoader(dev_ds, batch_size=16, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=0)

print(f"train={len(train_ds)} dev={len(dev_ds)} test={len(test_ds)}")

train=11811 dev=2361 test=3542


## 6. Train the fusion head

Plain PyTorch training loop over only the unfrozen parameters (`control_model.trainable_parameters()`).
Loss is `SmoothL1Loss` per output channel, with steering weighted higher since it's more
safety-critical than throttle/brake magnitude. Best checkpoint (lowest dev loss) is saved to
`artifacts/yolop_control_head_best.pt`.

In [16]:
import os
from pathlib import Path

NUM_EPOCHS = 10
CHANNEL_WEIGHTS = torch.tensor([1.0, 1.0, 2.0], device=device)  # throttle, brake, steer

optimizer = torch.optim.AdamW(control_model.trainable_parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = torch.nn.SmoothL1Loss(reduction="none")

Path("artifacts").mkdir(exist_ok=True)
best_dev_loss = float("inf")
CHECKPOINT_PATH = "artifacts/yolop_control_head_best.pt"


def run_epoch(loader, train: bool):
    control_model.numeric_branch.train(train)
    control_model.fusion_head.train(train)
    total_loss, n = 0.0, 0
    for images, numeric, targets in loader:
        images, numeric, targets = images.to(device), numeric.to(device), targets.to(device)
        with torch.set_grad_enabled(train):
            preds = control_model(images, numeric)
            loss = (loss_fn(preds, targets) * CHANNEL_WEIGHTS).mean()
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * images.size(0)
        n += images.size(0)
    return total_loss / n


for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = run_epoch(train_loader, train=True)
    dev_loss = run_epoch(dev_loader, train=False)
    print(f"epoch {epoch:02d}  train_loss={train_loss:.4f}  dev_loss={dev_loss:.4f}")

    if dev_loss < best_dev_loss:
        best_dev_loss = dev_loss
        torch.save({
            "numeric_branch": control_model.numeric_branch.state_dict(),
            "fusion_head": control_model.fusion_head.state_dict(),
            "encoder_layer_index": control_model.ENCODER_LAYER_INDEX,
            "dev_loss": dev_loss,
        }, CHECKPOINT_PATH)
        print(f"  saved new best checkpoint (dev_loss={dev_loss:.4f})")

c:\Users\nalli\Documents\Workspace Uni\TP FINAL Vision\TP-final-Vision-Artificial\.venv\Lib\site-packages\torch\functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4384.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


epoch 01  train_loss=0.0023  dev_loss=0.0010
  saved new best checkpoint (dev_loss=0.0010)
epoch 02  train_loss=0.0013  dev_loss=0.0010
  saved new best checkpoint (dev_loss=0.0010)
epoch 03  train_loss=0.0012  dev_loss=0.0009
  saved new best checkpoint (dev_loss=0.0009)
epoch 04  train_loss=0.0011  dev_loss=0.0009
  saved new best checkpoint (dev_loss=0.0009)
epoch 05  train_loss=0.0011  dev_loss=0.0008
  saved new best checkpoint (dev_loss=0.0008)
epoch 06  train_loss=0.0011  dev_loss=0.0009
epoch 07  train_loss=0.0011  dev_loss=0.0009
epoch 08  train_loss=0.0010  dev_loss=0.0008
  saved new best checkpoint (dev_loss=0.0008)
epoch 09  train_loss=0.0010  dev_loss=0.0008
  saved new best checkpoint (dev_loss=0.0008)
epoch 10  train_loss=0.0010  dev_loss=0.0008


## 7. Evaluate on the test split

Reload the best checkpoint and report per-channel MAE both in normalized (BeamNG) units and
in human-interpretable units (steering degrees-equivalent, acceleration m/s²-equivalent) via
the inverse transforms in `comma_beamng_transforms.py`.

In [17]:
import numpy as np
from comma_beamng_transforms import beamng_controls_to_accel, beamng_steering_to_deg

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
control_model.numeric_branch.load_state_dict(checkpoint["numeric_branch"])
control_model.fusion_head.load_state_dict(checkpoint["fusion_head"])

control_model.numeric_branch.eval()
control_model.fusion_head.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for images, numeric, targets in test_loader:
        preds = control_model(images.to(device), numeric.to(device))
        all_preds.append(preds.cpu().numpy())
        all_targets.append(targets.numpy())

preds = np.concatenate(all_preds)
targets = np.concatenate(all_targets)
mae = np.abs(preds - targets).mean(axis=0)

print("Per-channel MAE (normalized BeamNG units):")
print(f"  throttle: {mae[0]:.4f}  (0..1)")
print(f"  brake:    {mae[1]:.4f}  (0..1)")
print(f"  steer:    {mae[2]:.4f}  (-1..1)")

accel_pred = beamng_controls_to_accel(preds[:, 0], preds[:, 1])
accel_true = beamng_controls_to_accel(targets[:, 0], targets[:, 1])
steer_deg_pred = beamng_steering_to_deg(preds[:, 2])
steer_deg_true = beamng_steering_to_deg(targets[:, 2])

print("\nPer-channel MAE (human units):")
print(f"  longitudinal accel: {np.abs(accel_pred - accel_true).mean():.3f} m/s^2")
print(f"  steering angle:     {np.abs(steer_deg_pred - steer_deg_true).mean():.2f} deg")

Per-channel MAE (normalized BeamNG units):
  throttle: 0.0261  (0..1)
  brake:    0.0583  (0..1)
  steer:    0.0078  (-1..1)

Per-channel MAE (human units):
  longitudinal accel: 0.523 m/s^2
  steering angle:     3.89 deg


## 8. Inference sanity check

Spot-check a handful of test examples: predicted vs. ground-truth throttle/brake/steer.

In [18]:
N_SAMPLES = 5
sample_idx = np.random.choice(len(test_ds), size=min(N_SAMPLES, len(test_ds)), replace=False)

for i in sample_idx:
    image, numeric, target = test_ds[i]
    with torch.no_grad():
        pred = control_model(image.unsqueeze(0).to(device), numeric.unsqueeze(0).to(device))[0].cpu().numpy()
    row = test_ds.df.iloc[i]
    print(f"{row['image_path']}")
    print(f"  current : throttle={numeric[0]:.2f} brake={numeric[1]:.2f} steer={numeric[2]:.2f} speed={numeric[3]:.1f}m/s")
    print(f"  target  : throttle={target[0]:.2f} brake={target[1]:.2f} steer={target[2]:.2f}")
    print(f"  pred    : throttle={pred[0]:.2f} brake={pred[1]:.2f} steer={pred[2]:.2f}")
    print()

Chunk_1_b0c9d2329ad1606b_2018-07-30--13-44-30_6\frame_000704.jpg
  current : throttle=0.00 brake=0.06 steer=-0.01 speed=27.2m/s
  target  : throttle=0.00 brake=0.11 steer=-0.01
  pred    : throttle=0.00 brake=0.11 steer=0.00

Chunk_1_b0c9d2329ad1606b_2018-08-03--10-35-16_16\frame_000084.jpg
  current : throttle=0.00 brake=0.07 steer=0.00 speed=0.0m/s
  target  : throttle=0.00 brake=0.05 steer=0.00
  pred    : throttle=0.00 brake=0.09 steer=-0.01

Chunk_1_b0c9d2329ad1606b_2018-07-30--13-44-30_6\frame_000211.jpg
  current : throttle=0.00 brake=0.15 steer=-0.00 speed=21.7m/s
  target  : throttle=0.00 brake=0.07 steer=0.00
  pred    : throttle=0.00 brake=0.15 steer=0.01

Chunk_1_b0c9d2329ad1606b_2018-08-03--10-35-16_13\frame_000589.jpg
  current : throttle=0.00 brake=0.13 steer=0.00 speed=18.8m/s
  target  : throttle=0.00 brake=0.14 steer=-0.00
  pred    : throttle=0.01 brake=0.16 steer=-0.00

Chunk_1_b0c9d2329ad1606b_2018-08-03--10-35-16_16\frame_001089.jpg
  current : throttle=0.00 brake

In [19]:
# Save the notebook, then shut down the PC.
#
# NOTE on saving: this JS trick only works in classic Jupyter Notebook / JupyterLab
# frontends -- it does NOT reliably work in VS Code's notebook UI, which doesn't
# expose the same `Jupyter.notebook` JS object. If you're running this in VS Code,
# turn on "Files: Auto Save" (or just hit Ctrl+S yourself) instead of relying on
# this cell to persist your outputs.
from IPython.display import Javascript, display

display(Javascript("""
if (window.Jupyter && Jupyter.notebook) {
    Jupyter.notebook.save_checkpoint();
} else if (window.require) {
    require(['base/js/namespace'], function(Jupyter) { Jupyter.notebook.save_checkpoint(); });
}
"""))

import time
time.sleep(5)  # give the save a moment before shutting down

import subprocess
print("Shutting down in 60s. To cancel, open a terminal and run: shutdown /a")
subprocess.run([
    "shutdown", "/s", "/t", "60",
    "/c", "Notebook run finished - shutting down (run 'shutdown /a' to cancel)",
])

<IPython.core.display.Javascript object>

Shutting down in 60s. To cancel, open a terminal and run: shutdown /a


CompletedProcess(args=['shutdown', '/s', '/t', '60', '/c', "Notebook run finished - shutting down (run 'shutdown /a' to cancel)"], returncode=0)

## Next step (not done in this notebook)

`comma_beamng_transforms.py` and `artifacts/yolop_control_head_best.pt` are what
`yolop_beamng.py` would import to actually drive: read `electrics['wheelSpeed']` /
`electrics['steering']` and the last applied `throttle`/`brake` as the numeric input, run
`YOLOPControlNet`, and pass the output straight to `vehicle.control(steering=..., throttle=...,
brake=...)`. Wiring that into the live BeamNG loop — alongside or instead of the classical
PID-based LKA/ACC controller described in `plan.md` — is a follow-up, separate task.